Installing packages

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold,train_test_split
from sklearn.metrics import mean_absolute_error
import time
from itertools import product
import os
import joblib

from sklearn.preprocessing import StandardScaler


Loading training ECIF descriptors file

In [ ]:

df = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv") #use merge training descriptor file here in merge case


X = df.drop(columns=["Complex_ID", "pIC"]).values.astype(np.float32)
y = df["pIC"].values.astype(np.float32)


In [ ]:
df.shape

In [ ]:
df.head()

Loading previously made and saved training data split file

In [ ]:
import joblib

# Path to the saved file
load_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/"
load_path = os.path.join(load_dir, "ecif_splits_train.pkl")

# Load and unpack into same variables
X_train, X_test, y_train, y_test = joblib.load(load_path)

print("✅ Splits loaded successfully!")
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Scaling the training set and fitting on validation set

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)


Hyperparameter Tuning of XGBoost model

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Step 1: Define model (GPU version)
xgb_model = xgb.XGBRegressor(tree_method = "hist", device = "cuda",random_state=42)

# Step 2: Define hyperparameters to tune
n_estimators = list(range(1,151))   
max_depth = list(range(1,7))
learning_rate = [0.01, 0.05, 0.1]
gamma = [0, 0.5, 1]
min_child_weight = list(range(1,5))

# Step 3: Create the parameter grid for GridSearchCV
tuning_parameters = dict(
    n_estimators=n_estimators,
    max_depth=max_depth,
    learning_rate=learning_rate,
    gamma=gamma,
    min_child_weight=min_child_weight
)

# Step 4: Setup GridSearchCV
grid_search = GridSearchCV(estimator=xgb_model,param_grid=tuning_parameters,cv=5,scoring="neg_mean_absolute_error", n_jobs=-1,verbose=2)

# Step 5: Fit on training data
grid_search.fit(X_train, y_train)

# Step 6: Results
print("✅ Best parameters:", grid_search.best_params_)
print("✅ Best CV score:", grid_search.best_score_)


Traning XGBoost model

In [ ]:
#XGBOOST
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
import xgboost as xgb
model_xgb = xgb.XGBRegressor(n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    gamma=0,
    min_child_weight=1,
    random_state=42
)
model_xgb.fit(X_train, y_train)
y_pred = model_xgb.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
pcc, _ = pearsonr(y_test, y_pred)
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Pearson Correlation: {pcc:.3f}")

Saving model and scaler

In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/XGB_ECIF"
os.makedirs(save_dir, exist_ok=True)
joblib.dump(model_xgb, os.path.join(save_dir,"model.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))
df_full = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv")
features_name= df_full.drop(columns= ["Complex_ID", "pIC"]).columns.tolist()
with open(os.path.join(save_dir, "features.json"),"w") as f:
    json.dump(features_name,f)


print("everything saved")

Saving true values vs predicted values of validation set 

In [ ]:
#saving the results of validation set X_test too
# Create a dataframe for results
results_df = pd.DataFrame({
   
    "True_Value": y_test,
    "Predicted_Value": y_pred
})

# Save to CSV
results_df.to_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/val_pred.csv", index=False)

print("Validation predictions saved successfully ✅")

Plotting the true vs pred values of validation set

In [ ]:
import os
import joblib
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Define your model folder ===
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/XGB_ECIF"

# === Load saved validation predictions ===
val_df = pd.read_csv(os.path.join(save_dir, "val_pred.csv"))
y_val = val_df["True_Value"]
y_pred = val_df["Predicted_Value"]

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_val, y=y_pred)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')  # perfect fit line
plt.xlabel("Actual Binding Energy")
plt.ylabel("Predicted Binding Energy")
plt.title("Predicted vs Actual (XGBoost)")
plt.show()

plt.savefig(os.path.join(save_dir, "pred_vs_actual_plot.png"), dpi=300, bbox_inches='tight')

plt.show()

Loading saved xgboost model

In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/XGB_ECIF"
model_xgb = joblib.load(os.path.join(save_dir, "model.pkl"))
print(model_xgb.get_params())

Predicted pIC values of external dataset

In [ ]:
# Load everything
import joblib
import json
import os
import pandas as pd
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/XGB_ECIF"
model_xgb = joblib.load(os.path.join(save_dir, "model.pkl"))
scaler = joblib.load(os.path.join(save_dir, "scaler.pkl"))

with open(os.path.join(save_dir, "features.json"), "r") as f:
    feature_names = json.load(f)F_descriptors_t

# Prepare blind data
blind_df = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv")

# Ensure same feature order
X_blind = blind_df[feature_names].values

# Apply saved scaler
X_blind_scaled = scaler.transform(X_blind)

# Predict
y_pred_blind = model_xgb.predict(X_blind_scaled)
print("Predictions on blind data:", y_pred_blind[:10])

results_blind = pd.DataFrame({
    "PDB_CODE": blind_df["Complex_ID"],   
    "Predicted_Value": y_pred_blind
})

results_path = os.path.join(save_dir, "xgb_blind_predictions_new.csv")
results_blind.to_csv(results_path, index=False)

print(f"✅ Blind predictions saved at: {results_path}")
print(results_blind.head())

Hyperparameter Tuning of Random Forest model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# Step 1: Define model
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)
# Step 2: Define hyperparameters to tune
n_estimators = list(range(1,151))      # number of trees
max_depth = list(range(1,7))                 # tree depth             
min_samples_split = list(range(1,10))    # min samples to split a node
min_samples_leaf = list(range(1,5))            # min samples at leaf

# Step 3: Create the parameter grid
tuning_parameters = dict(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf
)


# Step 4: Setup GridSearchCV
grid_search = GridSearchCV(rf_model,tuning_parameters,cv=5,scoring="neg_mean_absolute_error", n_jobs=-1,verbose=2)

# Step 5: Fit on training data
grid_search.fit(X_train, y_train)

# Step 6: Results
print("✅ Best parameters:", grid_search.best_params_)
print("✅ Best CV score:", grid_search.best_score_)


In [ ]:
print("✅ Best parameters:", grid_search.best_params_)
print("✅ Best CV score:", grid_search.best_score_)


Training RF model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
model_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=6,
    min_samples_split=6,
    min_samples_leaf=1,
    random_state=42,
   
)

# Fit
model_rf.fit(X_train, y_train)

# Predict
y_pred = model_rf.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
pcc, _ = pearsonr(y_test, y_pred)

print("Random Forest Evaluation")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Pearson Correlation: {pcc:.3f}")

In [ ]:
import numpy as np

# Create mask for non-NaN targets
mask = ~np.isnan(y_test)

# Apply mask (NumPy-style)
X_test = X_test[mask]
y_test = y_test[mask]


Saving model and scaler

In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/RF_ECIF"
os.makedirs(save_dir, exist_ok=True)
joblib.dump(model_rf, os.path.join(save_dir,"model.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))
df_full = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv")
features_name= df_full.drop(columns= ["Complex_ID", "pIC"]).columns.tolist()
with open(os.path.join(save_dir, "features.json"),"w") as f:
    json.dump(features_name,f)


print("everything saved")

Saving true values vs predicted values of validation set 

In [ ]:
#saving the results of validation set X_test too
# Create a dataframe for results
results_df = pd.DataFrame({
   
    "True_Value": y_test,
    "Predicted_Value": y_pred
})

# Save to CSV
results_df.to_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/RF_ECIF/val_pred.csv", index=True)

print("Validation predictions saved successfully ✅")

Plotting the true vs pred values of validation set

In [ ]:
import os
import joblib
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Define your model folder ===
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/RF_ECIF"

# === Load saved validation predictions ===
val_df = pd.read_csv(os.path.join(save_dir, "val_pred.csv"))
y_val = val_df["True_Value"]
y_pred = val_df["Predicted_Value"]

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_val, y=y_pred)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')  # perfect fit line
plt.xlabel("Actual Binding Energy")
plt.ylabel("Predicted Binding Energy")
plt.title("Predicted vs Actual (RF)")
plt.show()


Predicted pIC values of external dataset

In [ ]:
import joblib
import json
import os
import pandas as pd
# Load everything
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/RF_ECIF"
model_rf = joblib.load(os.path.join(save_dir, "model.pkl"))
scaler = joblib.load(os.path.join(save_dir, "scaler.pkl"))

with open(os.path.join(save_dir, "features.json"), "r") as f:
    feature_names = json.load(f)

# Prepare blind data
blind_df = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv")

# Ensure same feature order
X_blind = blind_df[feature_names].values

# Apply saved scaler
X_blind_scaled = scaler.transform(X_blind)

# Predict
y_pred_blind = model_rf.predict(X_blind_scaled)
print("Predictions on blind data:", y_pred_blind[:10])

results_blind = pd.DataFrame({
    "PDB_CODE": blind_df["Complex_ID"],   
    "Predicted_Value": y_pred_blind
})

results_path = os.path.join(save_dir, "rf_blind_predictions_new.csv")
results_blind.to_csv(results_path, index=False)

print(f"✅ Blind predictions saved at: {results_path}")
print(results_blind.head())

Hyperparameter Tuning of Decision Tree model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

# Step 1: Define model
dt_model = DecisionTreeRegressor(random_state=42)
# Step 2: Define hyperparameters to tune

max_depth = list(range(1,7))                 # tree depth             
min_samples_split = list(range(1,10))    # min samples to split a node
min_samples_leaf = list(range(1,5))  
      
criterion=["squared_error","friedman_mse","absolute_error"]

# Step 3: Create the parameter grid
tuning_parameters = dict(
    criterion=criterion,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    learning_rate = [0.01,0.05, 0.1]  
)


# Step 4: Setup GridSearchCV
grid_search_dt = GridSearchCV(dt_model,tuning_parameters,cv=5,scoring="neg_mean_absolute_error", n_jobs=-1,verbose=2)

# Step 5: Fit on training data
grid_search_dt.fit(X_train, y_train)

# Step 6: Results
print("✅ Best parameters:", grid_search_dt.best_params_)
print("✅ Best CV score:", grid_search_dt.best_score_)


In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/DT_ECIF"
model_dt = joblib.load(os.path.join(save_dir, "model.pkl"))
print(model_dt.get_params())

Traning DT Model

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
from sklearn.tree import DecisionTreeRegressor
model_dt = DecisionTreeRegressor(
    criterion= 'absolute_error',
    max_depth=5,
    min_samples_split=9,
    min_samples_leaf=3,
    random_state=42,
   
)

# Fit
model_dt.fit(X_train, y_train)

# Predict
y_pred = model_dt.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
pcc, _ = pearsonr(y_test, y_pred)

print("decision tree Evaluation")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Pearson Correlation: {pcc:.3f}")

Saving model and scaler

In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/DT_ECIF"
os.makedirs(save_dir, exist_ok=True)
joblib.dump(model_dt, os.path.join(save_dir,"model.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))
df_full = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv")
features_name= df_full.drop(columns= ["Complex_ID", "pIC"]).columns.tolist()
with open(os.path.join(save_dir, "features.json"),"w") as f:
    json.dump(features_name,f)


print("everything saved")

Saving true values vs predicted values of validation set 

In [ ]:
#saving the results of validation set X_test too
# Create a dataframe for results
results_df = pd.DataFrame({
   
    "True_Value": y_test,
    "Predicted_Value": y_pred
})

# Save to CSV
results_df.to_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/DT_ECIF/val_pred.csv", index=True)

print("Validation predictions saved successfully ✅")

Plotting the true vs pred values of validation set

In [ ]:
import os
import joblib
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Define your model folder ===
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/DT_ECIF"

# === Load saved validation predictions ===
val_df = pd.read_csv(os.path.join(save_dir, "val_pred.csv"))
y_val = val_df["True_Value"]
y_pred = val_df["Predicted_Value"]

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_val, y=y_pred)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')  # perfect fit line
plt.xlabel("Actual Binding Energy")
plt.ylabel("Predicted Binding Energy")
plt.title("Predicted vs Actual (DT)")
plt.show()


Predicted pIC values of external dataset

In [ ]:
import joblib
import json
import os
import pandas as pd
# Load everything
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/DT_ECIF"
model_dt = joblib.load(os.path.join(save_dir, "model.pkl"))
scaler = joblib.load(os.path.join(save_dir, "scaler.pkl"))

with open(os.path.join(save_dir, "features.json"), "r") as f:
    feature_names = json.load(f)

# Prepare blind data
blind_df = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv")

# Ensure same feature order
X_blind = blind_df[feature_names].values

# Apply saved scaler
X_blind_scaled = scaler.transform(X_blind)

# Predict
y_pred_blind = model_dt.predict(X_blind_scaled)
print("Predictions on blind data:", y_pred_blind[:10])

results_blind = pd.DataFrame({
    "PDB_CODE": blind_df["Complex_ID"],   
    "Predicted_Value": y_pred_blind
})

results_path = os.path.join(save_dir, "dt_blind_predictions_new.csv")
results_blind.to_csv(results_path, index=False)

print(f"✅ Blind predictions saved at: {results_path}")
print(results_blind.head())

Hyperparameter Tuning of SVR model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

# Step 1: Define model
svm_model = SVR()

# Step 2: Define hyperparameters to tune
C = [0.1, 0.01, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
gamma = ["scale", "auto", 0.01, 0.1, 1, 0.001]
epsilon = [0.01, 0.1, 0.5, 1.0, 0.05, 0.07]

# Step 3: Create the parameter grid for GridSearchCV
tuning_parameters = dict(
    C=C,
    gamma=gamma,
    epsilon=epsilon
)

# Step 4: Setup GridSearchCV
grid_search_svr = GridSearchCV(svm_model,tuning_parameters,cv=5,scoring="neg_mean_absolute_error", n_jobs=-1,verbose=2)

# Step 5: Fit on training data
grid_search_svr.fit(X_train, y_train)

# Step 6: Results
print("✅ Best parameters:", grid_search_svr.best_params_)
print("✅ Best CV score:", grid_search_svr.best_score_)


Traning SVR Model

In [ ]:
from sklearn.svm import SVR
model_svr = SVR(
    C=4,
    gamma="scale",
    epsilon=0.01,

)

# Fit
model_svr.fit(X_train, y_train)

# Predict
y_pred = model_svr.predict(X_test)


# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
pcc, _ = pearsonr(y_test, y_pred)

print("SVR Evaluation")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Pearson Correlation: {pcc:.3f}")

Saving model and scaler

In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/SVR_ECIF"
os.makedirs(save_dir, exist_ok=True)
joblib.dump(model_svr, os.path.join(save_dir,"model.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))
df_full = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv")
features_name= df_full.drop(columns= ["Complex_ID", "pIC"]).columns.tolist()
with open(os.path.join(save_dir, "features.json"),"w") as f:
    json.dump(features_name,f)


print("everything saved")

Saving true values vs predicted values of validation set 

In [ ]:
#saving the results of validation set X_test too
# Create a dataframe for results
results_df = pd.DataFrame({
   
    "True_Value": y_test,
    "Predicted_Value": y_pred
})

# Save to CSV
results_df.to_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/SVR_ECIF/val_pred.csv", index=True)

print("Validation predictions saved successfully ✅")

Plotting the true vs pred values of validation set

In [ ]:
import os
import joblib
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Define your model folder ===
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/SVR_ECIF"


# === Load saved validation predictions ===
val_df = pd.read_csv(os.path.join(save_dir, "val_pred.csv"))
y_val = val_df["True_Value"]
y_pred = val_df["Predicted_Value"]

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_val, y=y_pred)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')  # perfect fit line
plt.xlabel("Actual Binding Energy")
plt.ylabel("Predicted Binding Energy")
plt.title("Predicted vs Actual (SVR)")
plt.show()


Predicted pIC values of external dataset

In [ ]:
import joblib
import json
import os
import pandas as pd
# Load everything
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/SVR_ECIF"
model_svr = joblib.load(os.path.join(save_dir, "model.pkl"))
scaler = joblib.load(os.path.join(save_dir, "scaler.pkl"))

with open(os.path.join(save_dir, "features.json"), "r") as f:
    feature_names = json.load(f)

# Prepare blind data
blind_df = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv")

# Ensure same feature order
X_blind = blind_df[feature_names].values

# Apply saved scaler
X_blind_scaled = scaler.transform(X_blind)

# Predict
y_pred_blind = model_svr.predict(X_blind_scaled)
print("Predictions on blind data:", y_pred_blind[:10])

results_blind = pd.DataFrame({
    "PDB_CODE": blind_df["Complex_ID"],   
    "Predicted_Value": y_pred_blind
})

results_path = os.path.join(save_dir, "SVR_blind_predictions_new.csv")
results_blind.to_csv(results_path, index=False)

print(f"✅ Blind predictions saved at: {results_path}")
print(results_blind.head())

Hyperparameter Tuning of Gradient Boosting Model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

# Step 1: Define model
gbr_model = GradientBoostingRegressor(random_state=42)
# Step 2: Define hyperparameters to tune
n_estimators = list(range(1,151))      # number of trees
max_depth = list(range(1,7))                 # tree depth             
min_samples_split = list(range(1,10))    # min samples to split a node
min_samples_leaf = list(range(1,5))  
learning_rate = [0.01,0.05, 0.1]       
     # min samples at leaf

# Step 3: Create the parameter grid
tuning_parameters = dict(
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf
)


# Step 4: Setup GridSearchCV
grid_search = GridSearchCV(gbr_model,tuning_parameters,cv=5,scoring="neg_mean_absolute_error", n_jobs=-1,verbose=2)

# Step 5: Fit on training data
grid_search.fit(X_train, y_train)

# Step 6: Results
print("✅ Best parameters:", grid_search.best_params_)
print("✅ Best CV score:", grid_search.best_score_)


In [ ]:
# Step 6: Results
print("✅ Best parameters:", grid_search.best_params_)
print("✅ Best CV score:", grid_search.best_score_)

Hyperparameter Tuning of Gradient Boosting model

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
from sklearn.ensemble import GradientBoostingRegressor
model_gbr = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=6,
    min_samples_split=7,
    min_samples_leaf=3,
    random_state=42,
   
)

# Fit
model_gbr.fit(X_train, y_train)

# Predict
y_pred = model_gbr.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
pcc, _ = pearsonr(y_test, y_pred)

print("GBR results")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Pearson Correlation: {pcc:.3f}")

Saving model and scaler

In [ ]:
import json
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/GBR_ECIF"
os.makedirs(save_dir, exist_ok=True)
joblib.dump(model_gbr, os.path.join(save_dir,"model.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))
df_full = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv")
features_name= df_full.drop(columns= ["Complex_ID", "pIC"]).columns.tolist()
with open(os.path.join(save_dir, "features.json"),"w") as f:
    json.dump(features_name,f)


print("everything saved")

Saving true values vs predicted values of validation set 

In [ ]:
#saving the results of validation set X_test too
# Create a dataframe for results
results_df = pd.DataFrame({
   
    "True_Value": y_test,
    "Predicted_Value": y_pred
})

# Save to CSV
results_df.to_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/GBR_ECIF/val_pred.csv", index=True)

print("Validation predictions saved successfully ✅")

Plotting the true vs pred values of validation set

In [ ]:
import os
import joblib
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Define your model folder ===
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/GBR_ECIF"

# === Load model and scaler ===
model_xgb = joblib.load(os.path.join(save_dir, "model.pkl"))
scaler = joblib.load(os.path.join(save_dir, "scaler.pkl"))

# === Load features (not strictly needed for plotting, but good to keep) ===
with open(os.path.join(save_dir, "features.json"), "r") as f:
    feature_names = json.load(f)

# === Load saved validation predictions ===
val_df = pd.read_csv(os.path.join(save_dir, "val_pred.csv"))
y_val = val_df["True_Value"]
y_pred = val_df["Predicted_Value"]

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,6))
sns.scatterplot(x=y_val, y=y_pred)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')  # perfect fit line
plt.xlabel("Actual Binding Energy")
plt.ylabel("Predicted Binding Energy")
plt.title("Predicted vs Actual (GBR)")
plt.show()


Predicted pIC values of external dataset

In [ ]:
import joblib
import json
import os
import pandas as pd
# Load everything
save_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/MODEL(ecif)/GBR_ECIF"
model_rf = joblib.load(os.path.join(save_dir, "model.pkl"))
scaler = joblib.load(os.path.join(save_dir, "scaler.pkl"))

with open(os.path.join(save_dir, "features.json"), "r") as f:
    feature_names = json.load(f)

# Prepare blind data
blind_df = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv")

# Ensure same feature order
X_blind = blind_df[feature_names].values

# Apply saved scaler
X_blind_scaled = scaler.transform(X_blind)

# Predict
y_pred_blind = model_rf.predict(X_blind_scaled)
print("Predictions on blind data:", y_pred_blind[:10])

results_blind = pd.DataFrame({
    "PDB_CODE": blind_df["Complex_ID"],   
    "Predicted_Value": y_pred_blind
})

results_path = os.path.join(save_dir, "gbr_blind_predictions_new.csv")
results_blind.to_csv(results_path, index=False)

print(f"✅ Blind predictions saved at: {results_path}")
print(results_blind.head())